# J3 — Comparer sans se mentir, puis ce qu’on fait des défauts trouvés

**Séance 3.** La chaîne, vous la connaissez : lire, explorer, entraîner, soumettre. Aujourd’hui,
elle sert deux fois. D’abord à **comparer** : un score, contre quoi ? Puis à **décider** de ce que
l’exploration a trouvé — les trous, les colonnes inutiles, les valeurs impossibles.

> **La règle :** une décision — de modèle ou de nettoyage — se **mesure**. Un effet nul est un
> résultat : il dit que cette décision-là ne se joue pas là.

**Les chiffres du texte** sont ceux des cellules : si une cellule en donne un autre, c’est la
cellule qui a raison.

**Le fil :** on ne nettoie pas ce qu’on ne sait pas mesurer. La **barre de bruit** — l’erreur-type
d’une exactitude sur le jeu de contrôle — est posée dans la première moitié. C’est elle qui rend
lisible le « le score ne bouge pas » de la seconde.

**Ce que le devoir 2 vous demandera :** les baselines et le tableau à quatre colonnes. Le journal
de nettoyage (décision · choix · effet mesuré). Et le **pipeline**, qui va de la table brute à la
prédiction.

> **Ce qui est évalué, et ce qui se lit.** Trois objets traversent tout le cours, et on vous
> interrogera dessus — en devoir, au partiel, à l’oral. Le premier est la fonction `predict` :
> l’interface que le professeur appelle sur une table brute. Le deuxième est le **tableau d’essais** :
> ce que vous avez annoncé, essayé, mesuré. Le troisième est le **journal de nettoyage** : chaque
> décision, son choix, son effet mesuré. Ce qu’ils **retournent** est ce qu’on vous demandera de
> défendre — et votre projet s’appuiera dessus.

**La table de travail n’est pas reconstruite ici** : la mise en place qui suit est **extraite de
`notebook_J1.ipynb`**, qui reste la source unique. Il n’existe qu’une seule définition de
`matrice()` dans tout le cours.


## Mise en place (fournie)

Ces cellules sont **extraites de `notebook_J1.ipynb`**, qui reste la source unique de la table
de travail, de la référence et du modèle. Exécutez-les de haut en bas.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5).
Le même nombre redonne le même découpage, donc la même mesure.


### Les outils


In [1]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [2]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"

print("le dossier des données trouvé :", DOSSIER)


le dossier des données trouvé : sortie


In [3]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"

print("les deux tableaux \n")
print("train :", train.shape, "(lignes, colonnes)")
print("test  :", test.shape, "(lignes, colonnes)")

print("les cinq premières lignes \n")
display(train.head())


les deux tableaux 

train : (149894, 39) (lignes, colonnes)
test  : (63721, 38) (lignes, colonnes)
les cinq premières lignes 

    dossier_id  ligne_id  ... taux_absence_passe patient_absent
0  DOS-0127912    127913  ...                0.0              1
1  DOS-0002883      2884  ...                0.0              0
2  DOS-0098561     98562  ...                0.0              0
3  DOS-0133156    133157  ...                0.0              0
4  DOS-0055744     55745  ...                0.0              0

[5 rows x 39 columns]


### La table de travail


In [4]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)

print("la table de travail \n")
print("colonnes conservées :", X.shape[1])
print("colonnes écartées (cardinalité trop forte) :", TROP_FIN)
print("colonnes catégorielles :", CAT)


la table de travail 

colonnes conservées : 26
colonnes écartées (cardinalité trop forte) : ['tournee_jour']
colonnes catégorielles : ['clinique', 'district', 'sexe', 'quartier', 'assurance', 'rappel_envoye', 'zone_clinique']


### La cible, et la référence


In [5]:
print("la distribution de la cible \n")
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])
display(repartition)

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())
print(f"la référence (répondre toujours la classe majoritaire) : {reference:.2f} %")


la distribution de la cible 

         lignes  part (%)
toutes   149894    100.00
absents   27357     18.25
la référence (répondre toujours la classe majoritaire) : 81.75 %


In [6]:
# `y` entre ici ; la référence a déjà été calculée avec la distribution ci-dessus —
y = train[CIBLE]


### Le rituel : annoncer avant de mesurer


In [7]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))


### Ce qu’un score doit battre

**82,51 %** ne dit pas qu’un modèle est bon. Cela dit qu’il fait un peu mieux que **81,75 %**,
la réponse constante. Un score ne prend son sens **que** contre un concurrent nommé. En production,
ce concurrent est rarement un autre modèle : c’est la **règle simple** en place. Elle tient en
trois lignes, personne ne la maintient, et personne n’ose la remplacer.

Ce qu’on regarde : **quatre** baselines mesurées sur le **même** jeu de validation. La réponse
constante. Une règle du délai. Une seconde règle, lue dans l’historique du patient.
La famille linéaire. Puis l’écart du modèle **à la réponse constante**, exprimé en barres de bruit.
Un écart plus petit que la barre n’est pas un gain : c’est du tirage.

Les valeurs essayées s’arrêtent avant le délai le plus long de la table. Au-delà, la règle
n’annoncerait plus rien — et ne rien annoncer donne la meilleure exactitude sur cette cible rare.

Une chose que la mesure montre : la règle du délai **perd** contre la réponse constante. Et la règle
qui ne dirait jamais rien la **vaudrait exactement**.

Pour plus tard : ce qu’une exactitude dit d’une cible rare.
La métrique de décision, posée plus bas, tranchera.


In [8]:
# les baselines, sur le même jeu de validation — et la barre qui décide
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

# 1) la réponse constante : le concurrent le plus bête, et le plus dur à battre
constante = 100 * max(yval.mean(), 1 - yval.mean())

# 2) la règle du délai : « j'annonce une absence si le délai dépasse N jours ».
#    N se choisit sur les lignes d'apprentissage, pas sur celles de contrôle.
delai_max = int(Xtr['delai_jours'].max())
taux_par_seuil = {s: float(((Xtr['delai_jours'] > s).astype(int) == ytr).mean())
                  for s in (0, 1, 3, 7, 14, 30) if s < delai_max}
seuil_appris = max(taux_par_seuil, key=taux_par_seuil.get)
regle = (Xval['delai_jours'] > seuil_appris).astype(int)

# et la règle qui ne dirait jamais rien ? c'est la réponse constante, au point près
silence = (Xval['delai_jours'] > delai_max).astype(int)

# 3) la famille linéaire : elle a besoin de nombres, on lui en prépare — la recette des
#    séances précédentes (newton-cholesky : le solveur qui converge sur cette table)
X_n = pd.get_dummies(X, columns=CAT).fillna(0)   # 0 = trou comblé ; le linéaire ne lit pas les NaN
Xtr_n, Xval_n = X_n.loc[Xtr.index], X_n.loc[Xval.index]
lineaire = LogisticRegression(solver='newton-cholesky').fit(Xtr_n, ytr)

# 4) le modèle du cours, celui des séances précédentes
modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)

# la barre : l'erreur-type de l'exactitude sur CE jeu de validation
barre = 100 * ((constante / 100) * (1 - constante / 100) / len(yval)) ** 0.5

baselines = pd.DataFrame([
    {'baseline': 'toujours présent',
     'exactitude (%)': round(constante, 2)},
    {'baseline': f'règle du délai — absent si le délai dépasse {seuil_appris} j',
     'exactitude (%)': round(100 * (regle == yval).mean(), 2)},
    {'baseline': 'absent si ce patient a déjà manqué un rendez-vous',
     'exactitude (%)': round(100 * ((Xval['taux_absence_passe'] > 0).astype(int) == yval).mean(), 2)},
    {'baseline': 'la famille linéaire',
     'exactitude (%)': round(100 * lineaire.score(Xval_n, yval), 2)},
    {'baseline': 'le modèle du cours',
     'exactitude (%)': round(100 * modele.score(Xval, yval), 2)},
])
display(baselines)

ecart = 100 * modele.score(Xval, yval) - constante
print(f'→ le modèle dépasse la réponse constante de {ecart:+.2f} point,')
print(f'  soit {ecart / barre:.1f} erreurs-types de ce jeu de validation ({barre:.2f} point).')
print(f'  la règle du délai, elle, fait {100 * (regle == yval).mean():.2f} % : elle alerte sur')
print(f'  {int(regle.sum())} lignes et perd contre la réponse constante.')
print(f'  et la règle qui ne dirait jamais rien ferait {100 * (silence == yval).mean():.2f} % —')
print('  exactement la réponse constante. Une règle en place se mesure avant d’être opposée')
print('  à un modèle ; ce que cela dit d’une exactitude reste ouvert.')


                                            baseline  exactitude (%)
0                                   toujours présent           81.75
1   règle du délai — absent si le délai dépasse 30 j           79.92
2  absent si ce patient a déjà manqué un rendez-vous           78.65
3                                la famille linéaire           82.21
4                                 le modèle du cours           82.51
→ le modèle dépasse la réponse constante de +0.76 point,
  soit 3.4 erreurs-types de ce jeu de validation (0.22 point).
  la règle du délai, elle, fait 79.92 % : elle alerte sur
  1572 lignes et perd contre la réponse constante.
  et la règle qui ne dirait jamais rien ferait 81.75 % —
  exactement la réponse constante. Une règle en place se mesure avant d’être opposée
  à un modèle ; ce que cela dit d’une exactitude reste ouvert.


> **Question →** `Devoir_maison_2.ipynb`, **question 1 (carnet J3)** — les baselines de votre main, l'écart divisé par la barre.


### Comment un modèle apprend : la perte

Un modèle ne s’entraîne pas à maximiser l’exactitude. Il ajuste ses paramètres pour diminuer un
nombre : la **perte**. Ce nombre mesure à quel point la probabilité annoncée s’écarte de la
vérité, et il n’apparaît jamais dans un rapport.

Le modèle ne rend donc pas un oui ou un non : il rend une **probabilité** (`predict_proba`). Un
**seuil** la transforme en décision (`predict`). Les deux objets sont différents, et la séance 4
en dépend.

**La perte et la métrique jouent deux rôles.** La perte est ce que `fit` diminue : elle est
visible pendant l’entraînement. La métrique est ce qu’on lit à la fin, pour comparer des modèles
sur des lignes qu’ils n’ont pas vues.

**Et si la métrique décidait ?** Elle ne pourrait pas guider : l’exactitude ne bouge que par
paliers, quand une ligne bascule du mauvais côté. La perte, elle, est **lisse** : chaque petite
variation de probabilité la déplace, et sa pente indique où aller.


In [9]:
# la métrique bouge par paliers, la perte se déplace sans à-coup
from sklearn.metrics import log_loss
import numpy as np
p_base = np.array([0.49, 0.60, 0.30, 0.55])
vraies = np.array([0, 1, 0, 1])
for delta in (0.0, 0.01, 0.02, 0.05):
    p2 = p_base + delta
    acc = ((p2 > 0.5).astype(int) == vraies).mean()
    perte = log_loss(vraies, np.clip(p2, 1e-6, 1 - 1e-6))
    print(f'  décalage {delta:+.2f} · exactitude {100 * acc:5.1f} % · log-loss {perte:.4f}')
print('→ l’exactitude ne bouge que par un saut ; la log-loss répond à chaque pas.')


  décalage +0.00 · exactitude 100.0 % · log-loss 0.5347
  décalage +0.01 · exactitude 100.0 % · log-loss 0.5346
  décalage +0.02 · exactitude  75.0 % · log-loss 0.5348
  décalage +0.05 · exactitude  75.0 % · log-loss 0.5372
→ l’exactitude ne bouge que par un saut ; la log-loss répond à chaque pas.


**Et pendant l’entraînement ?** On le voit en **reprenant le modèle par étapes** : le réglage
`warm_start=True` garde les arbres déjà construits, et on demande 5, puis 10, puis 20, puis 30
arbres. À chaque arrêt, la perte et les deux scores sont relevés. La figure superpose quatre
courbes sur **deux échelles** : la perte à gauche, les scores à droite.


In [10]:
# le modèle repris par étapes — la perte et les scores, relevés à chaque arrêt
import joblib
import matplotlib.pyplot as plt
progression = HistGradientBoostingClassifier(max_iter=5, warm_start=True,
                                              early_stopping=False, random_state=0)
trace = []
for etape in (5, 10, 20, 30):
    progression.set_params(max_iter=etape)      # plus d’arbres : il continue
    progression.fit(Xtr, ytr)
    perte_app = log_loss(ytr, progression.predict_proba(Xtr)[:, 1])
    perte_val = log_loss(yval, progression.predict_proba(Xval)[:, 1])
    score_app = 100 * progression.score(Xtr, ytr)
    score_val = 100 * progression.score(Xval, yval)
    trace.append({'arbres': etape,
                  'perte (apprentissage)': round(perte_app, 4),
                  'perte (validation)': round(perte_val, 4),
                  'score (apprentissage) %': round(score_app, 2),
                  'score (validation) %': round(score_val, 2)})
    print(f'  {etape:>2d} arbres · perte app {perte_app:.4f} · val {perte_val:.4f}'
          f' · score app {score_app:5.2f} % · val {score_val:5.2f} %')

trace = pd.DataFrame(trace)
display(trace)

fig, gauche = plt.subplots(figsize=(9.4, 4.2))
droite = gauche.twinx()
gauche.plot(trace['arbres'], trace['perte (apprentissage)'], color='#2b6cb0', linestyle='--', label='perte, apprentissage')
gauche.plot(trace['arbres'], trace['perte (validation)'], color='#2b6cb0', label='perte, validation')
droite.plot(trace['arbres'], trace['score (apprentissage) %'], color='#2f855a', linestyle='--', label='score, apprentissage')
droite.plot(trace['arbres'], trace['score (validation) %'], color='#2f855a', label='score, validation')
gauche.set_xlabel('arbres construits')
gauche.set_ylabel('perte (log-loss)', color='#2b6cb0')
droite.set_ylabel('score (%)', color='#2f855a')
gauche.set_title('deux échelles : la perte (gauche) et les scores (droite)')
lignes, etiquettes = gauche.get_legend_handles_labels()
l2, e2 = droite.get_legend_handles_labels()
gauche.legend(lignes + l2, etiquettes + e2, fontsize=8, loc='center right')
plt.tight_layout(); plt.show()
print('→ la perte descend encore quand les scores plafonnent, et l’écart entre les deux')
print('  courbes de score s’ouvre : le modèle épouse les données sans mieux prédire.')


   5 arbres · perte app 0.4350 · val 0.4372 · score app 81.75 % · val 81.75 %
  10 arbres · perte app 0.4192 · val 0.4232 · score app 82.29 % · val 82.17 %
  20 arbres · perte app 0.4074 · val 0.4150 · score app 82.70 % · val 82.52 %
  30 arbres · perte app 0.4023 · val 0.4136 · score app 82.84 % · val 82.50 %
   arbres  perte (apprentissage)  ...  score (apprentissage) %  score (validation) %
0       5                 0.4350  ...                    81.75                 81.75
1      10                 0.4192  ...                    82.29                 82.17
2      20                 0.4074  ...                    82.70                 82.52
3      30                 0.4023  ...                    82.84                 82.50

[4 rows x 5 columns]
<cell 10 notebook_J3.ipynb>:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
→ la perte descend encore quand les scores plafonnent, et l’écart entre les deux
  courbes de score s’ouvre : le modèle épouse les donn

> **La suite, en séance 4 :** la perte du linéaire et celle de l’arbre, comment chaque famille
> utilise la perte, et la lecture de chaque perte (log-loss, erreur au carré, impureté).


### Le test est sacré, et le piège de l’affichage

Un score affiché après un envoi se lit comme une mesure. C’est l’**affichage** — le tableau des
scores que la plateforme montre, calculé sur un **échantillon** de mille lignes. Sur mille
lignes, deux modèles qui valent la même chose n’obtiennent pas le même score. Choisir le premier
de l’affichage, c’est souvent choisir **le plus chanceux**. Le piège n’est pas d’envoyer : c’est de
**décider** sur ce que l’affichage montre.

Cette mesure a une **marge d’erreur**, et elle se calcule sans entraîner. Une exactitude est
une proportion : son écart-type vaut √(p(1−p)/n). La marge vaut **2 écarts-types** — environ
95 % des tirages sous la loi normale. Deux scores plus proches que cette marge ne se classent
pas : l’écart est du bruit, rien de plus.

| Lignes mesurées | Écart-type | Marge (± 2 écarts-types) |
|---|---|---|
| 1 000 | 1,22 pt | 2,44 pt |
| 3 000 | 0,71 pt | 1,42 pt |
| 29 979 | 0,22 pt | 0,44 pt |

Multiplier l’échantillon par 30 divise l’écart-type par √30 ≈ 5,5 : 1,22 pt devient 0,22 pt.
C’est la seule source de la différence, il n’y a rien d’autre derrière ces trois nombres.

Douze variantes étalées sur 0,4 pt tiennent donc toutes dans la marge d’un échantillon de
1 000 lignes : 2,44 pt. Et choisir le meilleur de douze mesures bruitées **biaise le choix**
vers le haut — le premier de la liste l’est souvent *grâce* au bruit.

Ce qu’on regarde : douze variantes jugées sur **1 000 lignes affichées**, puis re-mesurées sur
**3 000 lignes que personne n’a regardées**. Les deux classements sont-ils le même ? La cellule
prend **≈ 2 minutes** : c’est le prix de la démonstration, et il se paie une fois.

Pour plus tard : laquelle de ces douze variantes envoyer.
Choisir, c’est décider sur un échantillon.


In [11]:
# l’affichage, monté en petit : 1 000 lignes montrées, 3 000 lignes jamais regardées
import numpy as np

# le rituel du cours : on écrit ce qu'on attend avant de mesurer — sinon `noter` refuse la ligne
prevoir("le classement affiché",
        "le premier de l’affichage reste le premier des lignes jamais montrées",
        "l’affichage est un petit échantillon : son classement est du bruit")

Xdev, Xreste, ydev, yreste = train_test_split(X, y, test_size=4000, stratify=y, random_state=0)
Xpub, Xpriv, ypub, ypriv = train_test_split(Xreste, yreste, test_size=3000, stratify=yreste, random_state=0)

reglages = ((20, 3), (20, 5), (20, 10), (20, 31), (40, 5), (40, 10),
            (60, 5), (60, 31), (80, 10), (100, 5), (150, 10), (200, 31))
variantes = []
for max_iter, feuilles in reglages:
    m = HistGradientBoostingClassifier(max_iter=max_iter, max_leaf_nodes=feuilles,
                                       random_state=0).fit(Xdev, ydev)
    variantes.append({'variante': f'max_iter={max_iter} · feuilles={feuilles}',
                      'affiché (%)': round(100 * m.score(Xpub, ypub), 2),
                      'jamais montré (%)': round(100 * m.score(Xpriv, ypriv), 2)})

tableau = pd.DataFrame(variantes).sort_values('affiché (%)', ascending=False).reset_index(drop=True)
display(tableau)

premier, dernier = tableau.iloc[0], tableau.iloc[-1]
print(f'→ le premier de l’affichage : {premier["affiché (%)"]:.2f} % sur les lignes affichées, {premier["jamais montré (%)"]:.2f} % sur celles jamais montrées.')
print(f'  le dernier de l’affichage  : {dernier["affiché (%)"]:.2f} % sur les lignes affichées, {dernier["jamais montré (%)"]:.2f} % sur celles jamais montrées.')
correlation = float(np.corrcoef(tableau['affiché (%)'], tableau['jamais montré (%)'])[0, 1])
print(f'  corrélation entre les deux classements, sur ces douze variantes : {correlation:.3f}')
quatre_affiche = tableau.head(4)['variante'].tolist()
quatre_jamais_montre = pd.DataFrame(variantes).sort_values('jamais montré (%)', ascending=False).head(4)['variante'].tolist()
print(f'  parmi les quatre premiers de l’affichage, {len([v for v in quatre_jamais_montre if v in quatre_affiche])} sont dans les quatre premiers des lignes jamais montrées.')
erreur_type_affiche = 100 * (ypub.mean() * (1 - ypub.mean()) / len(ypub)) ** 0.5
etendue_affiche = tableau['affiché (%)'].max() - tableau['affiché (%)'].min()
print(f'  l erreur-type d une exactitude sur {len(ypub)} lignes : {erreur_type_affiche:.2f} point —')
print(f'  pour des variantes étalées sur {etendue_affiche:.2f} point. Le classement tient dans')
print('  le bruit de l échantillon qui le produit.')
print('→ un classement qui ne se retrouve pas d’un jeu à l’autre ne classe rien :')
print('  il trie le hasard, et l’on croit avoir choisi le meilleur modèle.')

# et la ligne d'essai : l'attendu était écrit avant, le mesuré arrive ici
noter("le classement affiché", "douze variantes, 1 000 lignes affichées",
      "le premier de l’affichage reste premier", float(tableau.loc[0, "jamais montré (%)"]))


                   essai  ...   écrit à
0  le classement affiché  ...  13:56:05

[1 rows x 4 columns]
                      variante  affiché (%)  jamais montré (%)
0    max_iter=20 · feuilles=10         81.7              82.67
1     max_iter=20 · feuilles=3         81.6              81.80
2    max_iter=40 · feuilles=10         81.6              82.67
3    max_iter=80 · feuilles=10         81.6              82.80
4   max_iter=150 · feuilles=10         81.6              82.80
5     max_iter=20 · feuilles=5         81.5              82.23
6     max_iter=40 · feuilles=5         81.4              82.57
7     max_iter=60 · feuilles=5         81.4              82.53
8    max_iter=60 · feuilles=31         81.4              82.80
9   max_iter=200 · feuilles=31         81.4              82.80
10   max_iter=20 · feuilles=31         81.3              82.63
11   max_iter=100 · feuilles=5         81.3              82.53
→ le premier de l’affichage : 81.70 % sur les lignes affichées, 82.67 % sur cel

> **Question →** `Devoir_maison_2.ipynb`, **question 2 (carnet J3)** — le classement affiché tient-il ?


### Un gain de 0,2 point est-il un gain ?

Le même modèle, entraîné deux fois, ne rend pas le même score : le découpage change, donc la
mesure change. La question n’est jamais « le score a-t-il monté ? », mais « **de combien** a-t-il
bougé, et ce changement fait-il plus que ce que le hasard fait tout seul ? ».

La barre du cours est l’**erreur-type d’une exactitude** : **± 0,22 point** sur 29 979 lignes de
validation. Un écart ne se lit qu’à partir du **double**, soit 0,44 — deux erreurs-types,
c’est-à-dire environ 95 % des tirages sous une loi normale. Ce n’est pas une « règle du cours »
sortie de nulle part : c’est la dispersion d’une mesure sur cet échantillon-là, et elle se calcule.

Ce qu’on regarde : cinq graines sur le même modèle, c’est-à-dire la dispersion que personne ne
choisit. Puis deux modèles comparés sur **un seul et même** découpage.

La dispersion des cinq tirages est **plus petite** que la barre, et c’est normal. Les cinq
découpages partagent les mêmes lignes, donc leurs jeux de contrôle se recouvrent. Ce ne sont pas
les mêmes grandeurs : la barre juge un écart à la réponse constante, pas un écart entre deux
modèles.

Pour plus tard : si deux modèles diffèrent vraiment.
La comparaison ligne par ligne (test de McNemar) est nommée, pas développée.


In [12]:
# cinq découpages, le même modèle : la dispersion que personne ne choisit
graines = []
for graine in range(5):
    X_appris, X_controle, y_appris, y_controle = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=graine)
    m = HistGradientBoostingClassifier(random_state=0).fit(X_appris, y_appris)
    graines.append(round(100 * m.score(X_controle, y_controle), 2))

ecart_type = float(np.std(graines, ddof=1))
print('les cinq scores :', graines)
print(f'  moyenne {np.mean(graines):.2f} · écart-type {ecart_type:.3f} · étendue {max(graines) - min(graines):.2f} point')
print(f'  à comparer à la barre d’une exactitude sur {len(yval)} lignes : {barre:.2f} point.')
print('  plus petite, et c’est normal : les cinq tirages partagent les mêmes lignes, donc leurs')
print('  jeux de contrôle se recouvrent — deux grandeurs différentes, qui ne jugent pas la même')
print('  chose.')

# et deux modèles, sur le même découpage — la seule comparaison honnête
# `early_stopping=False` : sans lui, l'arrêt anticipé borne les deux au même arbre.
court = HistGradientBoostingClassifier(max_iter=60, random_state=0, early_stopping=False).fit(Xtr, ytr)
long = HistGradientBoostingClassifier(max_iter=120, random_state=0, early_stopping=False).fit(Xtr, ytr)
ecart = 100 * long.score(Xval, yval) - 100 * court.score(Xval, yval)
print(f'  deux modèles ({court.n_iter_} arbres contre {long.n_iter_}), un seul découpage : {ecart:+.2f} point')
print(f'  la barre vaut {barre:.2f} point, son double {2 * round(barre, 2):.2f} — la règle du cours.')
print('→ un écart plus petit que la barre n’apprend rien : il faut le même découpage, et la barre.')


les cinq scores : [82.51, 82.52, 82.39, 82.42, 82.52]
  moyenne 82.47 · écart-type 0.062 · étendue 0.13 point
  à comparer à la barre d’une exactitude sur 29979 lignes : 0.22 point.
  plus petite, et c’est normal : les cinq tirages partagent les mêmes lignes, donc leurs
  jeux de contrôle se recouvrent — deux grandeurs différentes, qui ne jugent pas la même
  chose.
  deux modèles (60 arbres contre 120), un seul découpage : -0.04 point
  la barre vaut 0.22 point, son double 0.44 — la règle du cours.
→ un écart plus petit que la barre n’apprend rien : il faut le même découpage, et la barre.


> **Question →** `Devoir_maison_2.ipynb`, **question 3 (carnet J3)** — deux modèles, un seul découpage.


### Le tableau à quatre colonnes

C'est le tableau qui ferme la comparaison : **modèle · validation · affiché · écart**. Il ne sert
pas à désigner un vainqueur, il sert à lire l'écart entre ce que vous mesurez chez vous et ce que
l’affichage montre. Quand votre mesure monte et que la mesure affichée reste plate, le modèle n'a
pas progressé. C'est votre choix qui a suivi le hasard de l'échantillon visible.

Ce qu'on regarde : le tableau rempli avec ce qui est mesurable ici — la colonne de l’affichage
n'existe que pour un modèle **effectivement envoyé**.

Pour plus tard : quelle ligne déployer.
Le tableau lit un écart, il ne désigne pas un vainqueur.


In [13]:
# le tableau à quatre colonnes : les scores mesurés ici, et le score affiché relevé après l’envoi
lignes = []
for nom, decision in (('toujours présent', np.zeros(len(yval), dtype=int)),
                      (f'règle du délai (délai > {seuil_appris} j)', regle),
                      ('le modèle du cours', modele.predict(Xval))):
    lignes.append({'modèle': nom,
                   'validation (%)': round(100 * (decision == yval).mean(), 2)})
tableau_quatre = pd.DataFrame(lignes)
# la colonne de l’affichage ne se remplit que pour un modèle réellement envoyé : ici, un seul
tableau_quatre['affiché (%)'] = [None, None, 81.81]   # le seul score affiché du cours : celui de la séance 1
tableau_quatre['écart'] = (tableau_quatre['affiché (%)'] - tableau_quatre['validation (%)']).round(2)
display(tableau_quatre)
print('→ une colonne vide n’est pas un oubli : c’est un modèle qui n’a pas été envoyé.')
print('  Un écart sain est petit. Un écart qui grandit dit que le modèle a été choisi pour plaire à la mesure visible, pas pour être bon sur des données jamais vues.')


                          modèle  validation (%)  affiché (%)  écart
0               toujours présent           81.75          NaN    NaN
1  règle du délai (délai > 30 j)           79.92          NaN    NaN
2             le modèle du cours           82.51        81.81   -0.7
→ une colonne vide n’est pas un oubli : c’est un modèle qui n’a pas été envoyé.
  Un écart sain est petit. Un écart qui grandit dit que le modèle a été choisi pour plaire à la mesure visible, pas pour être bon sur des données jamais vues.


> **Question →** `Devoir_maison_2.ipynb`, **question 4 (carnet J3)** — votre tableau à quatre colonnes.


### Le plafond — ce que les données permettent

La séance précédente a laissé une question ouverte. On a entraîné un modèle, il bat la référence.
Jusqu’où peut-on monter ?

**Trois familles très différentes** — des arbres entraînés par corrections successives, un arbre
unique, un modèle linéaire — donnent ici presque le même chiffre. Ce n’est pas un hasard. Quand
trois familles qui ne partagent pas leur mécanisme plafonnent au même endroit, le plafond n’est
pas dans le modèle. **Il est dans les données, pour cette métrique** — et le savoir évite de chercher pendant
six mois ce qui n’existe pas.

Pour plus tard : si une autre famille ferait mieux.
Cette section mesure un plafond, elle n’épuise pas le catalogue.


In [14]:
# les trois familles : elles entrent ici
from sklearn.tree import DecisionTreeClassifier

# le découpage, la table d'indicatrices et le linéaire viennent de la section précédente :
# ce carnet n'en donne qu'une définition, et c'est elle qui sert partout.
print("trois familles, la même table, la même mesure \n")
scores = {}
for nom, famille, donnees in (
        ("arbres (boosting)", HistGradientBoostingClassifier(random_state=0), (Xtr, Xval)),
        ("un arbre unique", DecisionTreeClassifier(random_state=0, max_depth=8), (Xtr_n, Xval_n)),
        ("modèle linéaire", LogisticRegression(solver="newton-cholesky"), (Xtr_n, Xval_n))):
    a, b = donnees
    scores[nom] = 100 * famille.fit(a, ytr).score(b, yval)
    print(f"  {nom:<22} {scores[nom]:6.2f} %")

print("\nl'écart entre la meilleure et la moins bonne famille :",
      round(max(scores.values()) - min(scores.values()), 2), "point(s).")
print("→ le plafond n'est pas dans le modèle.")
print(f"  trois familles, un écart de {max(scores.values()) - min(scores.values()):.2f} point :")
print("  la marge d'erreur du jeu est plus grande que cet écart. Le plafond est dans les données.")


trois familles, la même table, la même mesure 

  arbres (boosting)       82.51 %
  un arbre unique         82.37 %
  modèle linéaire         82.21 %

l'écart entre la meilleure et la moins bonne famille : 0.3 point(s).
→ le plafond n'est pas dans le modèle.
  trois familles, un écart de 0.30 point :
  la marge d'erreur du jeu est plus grande que cet écart. Le plafond est dans les données.


> **Question →** `Devoir_maison_2.ipynb`, **question 5 (carnet J3)** — le plafond.


#### Et l’autre cause du plafond : des étiquettes fausses

Le plafond peut venir des données. Il peut aussi venir des **étiquettes elles-mêmes**. Personne
n’annote parfaitement, et un annotateur qui se trompe ne se trompe pas comme un modèle.

On le simule : on **retourne une part des étiquettes d’entraînement** (0 %, 15 %, 30 %), et on
mesure sur une validation **intacte**. Sinon on mesurerait le bruit, pas ce qu’il coûte. La
question n’est pas « le score baisse-t-il ? » mais « **sur quel modèle** le bruit mord-il ? ». La
réponse se lit sur deux colonnes : les deux modèles de la séance précédente cohabitent ici.

Pour plus tard : d’où viennent les étiquettes fausses.
Un carnet ne corrige pas l’annotation.


In [15]:
# le bruit d'étiquette : la même table, avec des étiquettes d'entraînement abîmées.
# On mesure sur une validation intacte — sinon on mesure le bruit, pas ce qu'il coûte.
brouille = []
for part in (0.0, 0.15, 0.30):
    if part:
        touchees = ytr.sample(frac=part, random_state=0).index
        y_appris = ytr.copy()
        y_appris.loc[touchees] = 1 - y_appris.loc[touchees]
    else:
        y_appris = ytr

    ligne = {"étiquettes brouillées": f"{int(100 * part)} %"}

    # l'ensemble d'arbres, celui de la séance précédente : il apprend peu — le bruit le touche peu
    m = HistGradientBoostingClassifier(random_state=0).fit(Xtr, y_appris)
    ligne["ensemble d'arbres (%)"] = round(100 * m.score(Xval, yval), 2)

    # un arbre seul, sans limite de profondeur : il apprend tout, y compris ce qu'on vient de
    # brouiller. C'est celui de la démonstration « un arbre seul apprend la table par cœur ».
    t = DecisionTreeClassifier(random_state=0).fit(Xtr_n, y_appris)
    ligne["arbre seul, sans limite (%)"] = round(100 * t.score(Xval_n, yval), 2)

    brouille.append(ligne)

print("la même table, des étiquettes abîmées à des degrés différents \n")
display(pd.DataFrame(brouille))
print("→ le bruit d'étiquette ne mord pas sur un modèle qui apprend peu : il mord sur celui qui")
print("  apprend tout. C'est un signe qui ne trompe pas — et il se lit sur deux colonnes.")


la même table, des étiquettes abîmées à des degrés différents 

  étiquettes brouillées  ensemble d'arbres (%)  arbre seul, sans limite (%)
0                   0 %                  82.51                        73.74
1                  15 %                  82.46                        66.73
2                  30 %                  82.18                        59.56
→ le bruit d'étiquette ne mord pas sur un modèle qui apprend peu : il mord sur celui qui
  apprend tout. C'est un signe qui ne trompe pas — et il se lit sur deux colonnes.


> **Question →** `Devoir_maison_2.ipynb`, **question 6 (carnet J3)** — le bruit d'étiquette, et le modèle qu'il abîme.


### Un trou n’est pas toujours un trou

Un trou dans une colonne n’a pas toujours la même valeur. Parfois la valeur manque **au hasard**.
Le trou ne dit rien de la ligne, il dit seulement qu’on ne l’a pas notée. Parfois, au contraire,
le fait qu’elle manque est en soi une information. Le combler, c’est effacer cette information.

La règle qui suit : **avant d’imputer, on regarde ce que le trou vaut.** C’est la décision la plus
fréquente du domaine. Les débutants la prennent à l’envers : ils comblent d’abord, ils regardent
ensuite.

Pour plus tard : quelle décision prendre pour une colonne.
Elle se tranche colonne par colonne.


#### Ce qu’imputer veut dire, et les familles d’imputation

**Imputer, c’est remplacer une valeur manquante par une valeur fabriquée**, pour que la table soit
utilisable. Le geste n’est jamais neutre : la valeur écrite n’a pas été mesurée. On inscrit dans
la table quelque chose que personne n’a observé.

Les familles, et ce qu’elles coûtent :

| Famille | Ce qu’on écrit | Où elle sert, dans cette table |
|---|---|---|
| **une constante** (sentinelle) | `-1`, `0`, « inconnu » | `constantes_relevees` : le trou reste une catégorie |
| **une statistique de la colonne** | la moyenne, la **médiane** (robuste aux extrêmes), le **mode** | `solde_credit` (médiane) ; une catégorielle par son mode |
| **par groupe** | la statistique du groupe — par clinique, par jour | la table d’activité, en séance 6 |
| **par un modèle** | la valeur prédite depuis les autres colonnes | rarement nécessaire, et la plus chère |
| **rien : retirer** | la ligne, ou la colonne | `date_naissance` : 46 lignes sur 149 894 |

**Et une colonne qui accompagne toutes les autres : l’indicateur.** À côté de la valeur comblée, on
ajoute « cette ligne était vide » — 0 ou 1. Le fait qu’une valeur manque est souvent une
information ; l’indicateur la garde pendant que le trou disparaît.

Les cinq colonnes à trous de cette table, et la décision que chacune appelle :

| Colonne | Trous | Ce que le trou est | La décision |
|---|---|---|---|
| `motif_absence` | 81,75 % | la **réponse** elle-même | **on ne remplit pas** : c’est une fuite |
| `constantes_relevees` | 42,38 % | un état : « rien n’a été mesuré » | garder le trou comme catégorie |
| `rappel_envoye` | 66,27 % | à **mesurer** — aléatoire ou structuré ? | la mesure décide, colonne par colonne |
| `solde_credit` | 5,50 % | un montant, sans raison de croire au hasard | médiane **+ indicateur** |
| `date_naissance` | 0,03 % | 46 lignes sur 149 894 | les retirer, ou la valeur la plus simple |

**Deux règles gouvernent toutes les familles.** Un : la statistique se calcule sur la **part
d’apprentissage seulement** — la calculer sur toute la table fait entrer l’information du contrôle
dans la colonne. Deux : l’**indicateur** se garde, même quand la valeur comblée semble bonne.

Pour plus tard : quelle famille choisir pour une colonne.
La mesure dit laquelle.


#### La méthode : ce que vaut un trou, avant de le combler

On ne devine pas si un trou est aléatoire : on le **mesure**. Pour une colonne, on compare le
taux d’absence des lignes où elle **manque** à celui des lignes où elle est **remplie**. S’ils se
ressemblent, le trou ne dit rien de la réponse : on le traite comme **non informatif**. S’ils
s’écartent, le trou porte de l’information — les manuels parlent de **MNAR** (*missing not at
random*), et ce carnet dit un **trou informatif** : c’est cela qui sert à écrire la décision.

Deux règles avant de comparer. Un : **on ne compare que des groupes d’au moins 500 lignes**, parce
qu’un groupe de trente lignes a un taux qui bouge tout seul. Deux : on nomme **le sens** du taux
qu’on cite.

Les deux taux se lisent dans **deux sens différents**, et c’est là que les débutants se trompent.
« Chez les absents, 75 % des lignes ont un trou » part de la réponse pour aller vers le trou. C’est
vrai, et c’est inutilisable : au moment de prédire, on ne sait pas encore qui est absent.

« Parmi les lignes à trou, 32 % sont absentes » part du trou pour aller vers la réponse. C’est le
sens utile : le trou, lui, est visible avant la prédiction. Et 32 % se compare au taux global
d’absences : le trou fait presque doubler le risque. Voilà pourquoi il porte de l’information.


« Parmi les jours de pluie, combien sortent un parapluie » n’est pas la question inverse.
L’autre question est « parmi ceux qui sortent un parapluie, combien de jours il pleut ».
Le premier taux frôle 100 % ; le second dépend du climat. Et c’est le second qui décide : on voit
le parapluie avant de savoir le temps.
Attention aussi : **structuré ne veut pas dire fuite**. Un trou structuré dont l’origine n’est pas
la réponse est une information légitime, qu’on garde.


In [16]:
# les deux sens, mesurés sur la colonne du trou structuré
masque_trou = train["constantes_relevees"].isna()
masque_absent = train[CIBLE] == 1

n_absents, n_presents = int(masque_absent.sum()), int((~masque_absent).sum())
n_trous = int(masque_trou.sum())
trou_et_absent = int((masque_trou & masque_absent).sum())
trou_et_present = int((masque_trou & ~masque_absent).sum())

print("les deux sens, sur la colonne des constantes relevées \n")
print(f"  le taux global d’absence : {100 * masque_absent.mean():5.2f} %  ({n_absents} lignes sur {len(train)})")
print(f"  le taux global de trou   : {100 * masque_trou.mean():5.2f} %  ({n_trous} lignes)")
print()
print("  sens 1 — on part de la réponse, et on regarde le trou")
print(f"    chez les absents  : {100 * trou_et_absent / n_absents:5.2f} %  ({trou_et_absent} sur {n_absents})")
print(f"    chez les présents : {100 * trou_et_present / n_presents:5.2f} %  ({trou_et_present} sur {n_presents})")
print("    → le trou suit la réponse — mais au moment de prédire, la réponse n’est pas connue")
print()
print("  sens 2 — on part du trou, et on regarde la réponse")
print(f"    absents parmi les lignes à trou : {100 * trou_et_absent / n_trous:5.2f} %  ({trou_et_absent} sur {n_trous})")
print(f"    le taux global, pour comparer   : {100 * masque_absent.mean():5.2f} %")
print("    → c’est le sens qui sert : le trou est visible avant la prédiction")
print()
print(f"  recoupement : {100 * trou_et_absent / n_absents:.2f} % × {100 * masque_absent.mean():.2f} % / {100 * masque_trou.mean():.2f} % = {100 * trou_et_absent / n_trous:.2f} %")


les deux sens, sur la colonne des constantes relevées 

  le taux global d’absence : 18.25 %  (27357 lignes sur 149894)
  le taux global de trou   : 42.38 %  (63530 lignes)

  sens 1 — on part de la réponse, et on regarde le trou
    chez les absents  : 74.65 %  (20423 sur 27357)
    chez les présents : 35.18 %  (43107 sur 122537)
    → le trou suit la réponse — mais au moment de prédire, la réponse n’est pas connue

  sens 2 — on part du trou, et on regarde la réponse
    absents parmi les lignes à trou : 32.15 %  (20423 sur 63530)
    le taux global, pour comparer   : 18.25 %
    → c’est le sens qui sert : le trou est visible avant la prédiction

  recoupement : 74.65 % × 18.25 % / 42.38 % = 32.15 %


In [17]:
# la méthode, sur les deux colonnes que cette section traite
def etat_du_trou(table, colonne, plancher=500):
    """Les deux taux d'absence et les deux effectifs — refusés sous le plancher du cours."""
    manque = table[colonne].isna()
    n_manque, n_rempli = int(manque.sum()), int((~manque).sum())
    if min(n_manque, n_rempli) < plancher:
        return None
    return {'colonne': colonne,
            'n si ça manque': n_manque, 'n si c’est rempli': n_rempli,
            'taux d’absence si manque (%)': round(100 * table.loc[manque, CIBLE].mean(), 2),
            'taux d’absence si rempli (%)': round(100 * table.loc[~manque, CIBLE].mean(), 2)}

a_trous = [c for c in train.columns if train[c].isna().any()]
trop_fins = [c for c in a_trous if min(int(train[c].isna().sum()), int(train[c].notna().sum())) < 500]
print(f"le taux d'absence global, pour situer : {100 * train[CIBLE].mean():.2f} % \n")
print("les colonnes écartées du tableau (un groupe sous le plancher de 500) :", trop_fins)
display(pd.DataFrame([etat_du_trou(train, c) for c in ("solde_credit", "constantes_relevees")]))
print("→ deux taux qui se ressemblent : trou non informatif ; deux taux qui s'écartent : trou")
print("  informatif — et l'imputer efface ce qu'il dit.")


le taux d'absence global, pour situer : 18.25 % 

les colonnes écartées du tableau (un groupe sous le plancher de 500) : ['date_naissance']
               colonne  ...  taux d’absence si rempli (%)
0         solde_credit  ...                         18.27
1  constantes_relevees  ...                          8.03

[2 rows x 5 columns]
→ deux taux qui se ressemblent : trou non informatif ; deux taux qui s'écartent : trou
  informatif — et l'imputer efface ce qu'il dit.


In [18]:
# un trou aléatoire : cinq traitements, et ce que chacun change
def score_de(Xe, ye=None):
    """Entraîne et mesure, toujours sur la même découpe."""
    ye = y if ye is None else ye
    Xtr_m, Xval_m, ytr_m, yval_m = train_test_split(Xe, ye, test_size=0.2, stratify=ye, random_state=0)
    return 100 * HistGradientBoostingClassifier(random_state=0).fit(Xtr_m, ytr_m).score(Xval_m, yval_m)

# le même découpage, les lignes retirées seulement côté apprentissage : le jeu de contrôle
# ne change pas, donc les deux scores se comparent vraiment — ce que `score_de` ne fait pas
def score_sans_lignes(masque):
    """Retire des lignes du seul côté apprentissage, et mesure sur le même jeu de contrôle."""
    garde = masque.loc[Xtr.index]
    return 100 * HistGradientBoostingClassifier(random_state=0).fit(
        Xtr[garde], ytr[garde]).score(Xval, yval)

colonne = "solde_credit"
mediane = X.copy()
mediane[colonne] = mediane[colonne].fillna(mediane[colonne].median())
avec_indicateur = mediane.copy()
avec_indicateur[colonne + "_etait_vide"] = X[colonne].isna().astype(int)
masque = X[colonne].notna()

print(f"la colonne {colonne}, cinq traitements du même trou \n")
cinq = [score_de(X), score_de(X.drop(columns=[colonne])), score_de(mediane),
        score_de(avec_indicateur), score_sans_lignes(masque)]
noms = ["laissé tel quel", "colonne retirée", "trou comblé par la médiane",
        "médiane + indicateur", "lignes à trou supprimées"]
for nom, valeur in zip(noms, cinq):
    print(f"  {nom:<26} {valeur:6.2f} %")
print(f"\n→ un trou aléatoire ne vaut rien : les cinq chiffres tiennent dans {max(cinq) - min(cinq):.2f} point.")


la colonne solde_credit, cinq traitements du même trou 

  laissé tel quel             82.51 %
  colonne retirée             82.46 %
  trou comblé par la médiane  82.43 %
  médiane + indicateur        82.43 %
  lignes à trou supprimées    82.47 %

→ un trou aléatoire ne vaut rien : les cinq chiffres tiennent dans 0.08 point.


In [19]:
# un trou structurel : le même geste, un autre résultat
constantes = train["constantes_relevees"].fillna("").astype("category")
# le mode est calculé, jamais écrit à la main : c’est le choix qu’on dit avoir fait
mode_complet = constantes.mode()[0]

X_garde = X.copy()
X_garde["constantes_relevees"] = constantes

X_comble = X.copy()
# combler par le mode fusionne le trou avec la valeur qui existe déjà le plus souvent :
# « non renseigné » et « complet » deviennent indistinguables — c’est le geste qui efface
X_comble["constantes_relevees"] = constantes.astype(str).replace("", mode_complet).astype("category")

print("la même colonne, trois traitements du même trou \n")
print(f"  le trou gardé comme catégorie  {score_de(X_garde):6.2f} %")
print(f"  le trou comblé par le mode ({mode_complet})  {score_de(X_comble):6.2f} %")
print(f"  la colonne retirée             {score_de(X):6.2f} %")

print("\n→ ici, garder le trou vaut plus que le combler. Le trou porte de l’information :")
print("  imputer sans regarder l’avait effacée — et c’est ce que cette section mesure.")


la même colonne, trois traitements du même trou 

  le trou gardé comme catégorie   83.23 %
  le trou comblé par le mode (complet)   82.60 %
  la colonne retirée              82.51 %

→ ici, garder le trou vaut plus que le combler. Le trou porte de l’information :
  imputer sans regarder l’avait effacée — et c’est ce que cette section mesure.


#### Pourquoi ce trou porte de l’information, et ce que l’imputation efface

Le trou est un **état de la ligne**, pas une valeur : « les constantes n’ont jamais été relevées
pour ce patient ». C’est un fait, et il est mesuré : **74,65 %** des patients absents portent ce
trou, contre **35,18 %** des présents. Le modèle peut donc lire « pas de mesure » comme un signe.
Et il le lit : garder le trou comme catégorie bat le fait de le combler.

Combler par le mode efface ce signe. La valeur écrite — le mode — existe déjà dans la colonne. Les
lignes comblées deviennent **indistinguables** des lignes où la mesure a réellement été prise et
valait ce mode. La ligne ne dit plus « rien n’a été mesuré » ; elle dit « mesuré, et égal au mode ».
L’information n’est pas perdue par maladresse : elle est **effacée par construction**, et c’est ce
que la cellule ci-dessus chiffre.


#### Ce que ce trou est vraiment : une fuite

D’où vient ce trou ? Le générateur de cette table répond : « manquant structurel causé par la
cible ». **75 %** de trous chez les absents, **35 %** chez les présents. Le trou suit donc la
**réponse**. Et une colonne dont le trou suit la réponse est une **fuite** — la même famille que
`motif_absence`, en plus discret.

Comme le trou est produit dans les **deux** tables, elle ne se contente pas de gagner en interne :
elle gagne **aussi** sur la mesure affichée. C’est la seule fuite du cours qui **survit** au découpage.
Le contraste avec `canal_prise`, qui gagne en interne et perd sur la mesure affichée, est la leçon. Ce qui
la trahit n’est pas sa stabilité : c’est l’**origine** de son trou.


**Une colonne qui suit la cible n’est pas forcément une fuite.** Chercher les colonnes qui portent
du signal, c’est le travail du modèle. Une colonne corrélée à la cible est exactement ce qu’on
espère trouver. Ce qui fait la fuite, c’est **l’origine** de la valeur : elle n’existe que **parce
que** la réponse est connue.

Le test tient en une question de calendrier : **au moment où l’on prédit, cette valeur est-elle
disponible ?** `delai_jours` et `taux_absence_passe` sont connus avant la décision. Corrélés à la
cible, et parfaitement légitimes. Ici, le trou de `constantes_relevees` est **produit par la
réponse** — le générateur le dit : « manquant structurel causé par la cible ». Sur un nouveau
patient, au moment de décider, ce trou ne serait pas là.

**La décision, donc :** cette colonne est **écartée** — comme la table de travail le fait dès la
mise en place. Le **+0,72 point** qu’elle rapporte quand on la garde n’est pas un gain. C’est la
mesure de ce qu’une fuite rapporte, et **ce chiffre n’est pas fiable**.

Pour plus tard : si d’autres colonnes cachent la réponse.
C’est la traque du devoir, signe par signe.


> **Question →** `Devoir_maison_2.ipynb`, **question 7 (carnet J3)** — un trou aléatoire, un trou structuré.


### Les décisions qui ne se voient pas dans le score

Reste une famille de décisions, celles qu’on prend tous les jours sans se poser de question.
Supprimer une colonne constante. Écarter une colonne de bruit. Traiter un code sans documentation.
Retirer une ligne dupliquée. Retirer une valeur impossible.

On les mesure **toutes**, et le résultat est presque toujours le même : **le score ne bouge pas.**
Ce n’est pas une déception, c’est un résultat. **Le nettoyage change rarement le score. Il change
ce que vous pouvez défendre**, et ce qui survit quand les données changent.

Pour plus tard : lesquelles de ces décisions sont obligatoires.
Aucune ne se joue sur le score.


In [20]:
# les colonnes inutiles, et la raison de chacune
raisons = {
    "zone_clinique": "une seule valeur sur toute la table",
    "indice_meteo_jour": "du bruit pur",
    "score_territorial": "42 052 valeurs, aucune raison de croire qu’elles disent quelque chose",
    "age_mois": "age_mois ÷ 12 = age — sur 100 % des lignes (mesuré en J1, 2.6)",
    "cout_acte": "montant_acte = 13,68 × cout_acte, rapport constant (mesuré en J1, 2.6)",
}
inutiles = [c for c in raisons if c in X.columns]

print("les colonnes qui n’apportent rien \n")
print("  toutes les colonnes      ", f"{score_de(X):6.2f} %")
print(f"  sans {len(inutiles)} colonnes inutiles", f"{score_de(X.drop(columns=inutiles)):6.2f} %")
print("\n→ les colonnes écartées, et la raison de chacune :")
for c in inutiles:
    print(f"    {c:<20} {raisons[c]}")


les colonnes qui n’apportent rien 

  toutes les colonnes        82.51 %
  sans 5 colonnes inutiles  82.52 %

→ les colonnes écartées, et la raison de chacune :
    zone_clinique        une seule valeur sur toute la table
    indice_meteo_jour    du bruit pur
    score_territorial    42 052 valeurs, aucune raison de croire qu’elles disent quelque chose
    age_mois             age_mois ÷ 12 = age — sur 100 % des lignes (mesuré en J1, 2.6)
    cout_acte            montant_acte = 13,68 × cout_acte, rapport constant (mesuré en J1, 2.6)


In [21]:
# les catégories non documentées : un code arrive, personne ne sait ce qu’il vaut
if "code_assurance" in X.columns:
    codes = train["code_assurance"].value_counts().sort_index()
    inconnus = int(train["code_assurance"].isin([7, 9]).sum())
    print("les codes du régime, et ceux que le dictionnaire ne définit pas \n")
    print("  le dictionnaire publié décrit la colonne en une ligne — « code du régime » — et",
          "n’énumère pas ses valeurs ; il est livré avec les données, dans le dossier que ce carnet lit.")
    print("  codes présents :", ", ".join(f"{k} ({v} lignes)" for k, v in codes.items()))
    print(f"  dont non définis (7 et 9) : {inconnus} lignes ({100 * inconnus / len(train):.2f} %)")
    print()
    print("  colonne gardée    :", f"{score_de(X):6.2f} %")
    sans_code = score_de(X.drop(columns=['code_assurance']))
    print("  colonne retirée   :", f"{sans_code:6.2f} %")
    print("\n→ la décision ne se joue pas sur le score, mais sur ce qu’on peut dire de ce code :")
    print("  le garder en sachant qu’il est inexpliqué, le rattacher, ou l’écarter — et pourquoi.")


les codes du régime, et ceux que le dictionnaire ne définit pas 

  le dictionnaire publié décrit la colonne en une ligne — « code du régime » — et n’énumère pas ses valeurs ; il est livré avec les données, dans le dossier que ce carnet lit.
  codes présents : 1 (45654 lignes), 2 (36672 lignes), 3 (26139 lignes), 4 (18317 lignes), 5 (12133 lignes), 6 (9081 lignes), 7 (1343 lignes), 9 (555 lignes)
  dont non définis (7 et 9) : 1898 lignes (1.27 %)

  colonne gardée    :  82.51 %
  colonne retirée   :  82.50 %

→ la décision ne se joue pas sur le score, mais sur ce qu’on peut dire de ce code :
  le garder en sachant qu’il est inexpliqué, le rattacher, ou l’écarter — et pourquoi.


In [22]:
# les valeurs impossibles et les doublons
impossibles = train["age"] == -1
sans_doublons = train.drop_duplicates()

# les trois mesures sur le même découpage : `score_sans_lignes` ne retire que côté apprentissage
base_nettoyage = score_de(X)
sans_age = score_sans_lignes(~impossibles)
sans_double = score_sans_lignes(~train.drop(columns=[ID]).duplicated())

print("les valeurs impossibles, et les doublons \n")
print(f"  rien touché                  {base_nettoyage:6.2f} %")
print(f"  âges impossibles retirés     {sans_age:6.2f} %")
print(f"  lignes dupliquées retirées   {sans_double:6.2f} %")

print(f"\n→ la correction des âges retire {int(impossibles.sum())} lignes ; l’écart tient dans")
print(f"  {abs(base_nettoyage - sans_age):.2f} point, soit {abs(base_nettoyage - sans_age) / barre:.2f} barre — sous la barre, donc : le score ne bouge pas.")
print("  On jette de l’information avec le défaut — corriger n’est pas gratuit.")
print(f"  Les doublons, eux, ne retirent rien ({int(train.drop(columns=[ID]).duplicated().sum())} ligne) : le score")
print("  est identique, et c’est le résultat attendu — le défaut n’est pas dans cette table")
print("  (J1, 2.4). Une décision qui ne change rien se note quand même : « vérifié, rien à faire »")
print("  est une conclusion.")


les valeurs impossibles, et les doublons 

  rien touché                   82.51 %
  âges impossibles retirés      82.50 %
  lignes dupliquées retirées    82.51 %

→ la correction des âges retire 46 lignes ; l’écart tient dans
  0.00 point, soit 0.01 barre — sous la barre, donc : le score ne bouge pas.
  On jette de l’information avec le défaut — corriger n’est pas gratuit.
  Les doublons, eux, ne retirent rien (0 ligne) : le score
  est identique, et c’est le résultat attendu — le défaut n’est pas dans cette table
  (J1, 2.4). Une décision qui ne change rien se note quand même : « vérifié, rien à faire »
  est une conclusion.


> **Question →** `Devoir_maison_2.ipynb`, **question 8 (carnet J3) (1/2)** — les décisions qui ne se voient pas dans le score.


### Le journal de nettoyage, et le pipeline

Chaque décision est maintenant mesurée. On les rassemble dans **un tableau**, dans l’esprit du
tableau d’essais de la séance précédente : **décision · choix · effet mesuré**. C’est ce tableau
qu’on rend. C’est lui qui répond à la question d’un audit : *pourquoi avez-vous fait ça ?*

Puis on assemble : une seule fonction, de la table brute à la prédiction, qui porte les décisions de
colonnes ; les décisions de lignes — âges impossibles, doublons — se prennent avant, sur la table.
C’est le **pipeline**, et c’est lui que la suite exécute.
Pour plus tard : si ce pipeline est le bon découpage dans le temps.
Il ne l’est pas, et la dernière section le mesure.


In [23]:
# le journal de nettoyage — ce qui est lu n’est pas le score, c’est le tableau
JOURNAL = []

def journal(decision, choix, raison, effet):
    """Une décision, le choix fait, la raison qu’on donnerait à un audit, et l’effet mesuré
       en points — avec la comparaison au double de la barre, qui dit si l’effet compte."""
    JOURNAL.append({"décision": decision, "choix": choix, "raison (sans le score)": raison,
                    "effet mesuré": round(effet, 2),
                    "> double de la barre ?": "oui" if abs(effet) > 2 * barre else "non"})
    display(pd.DataFrame(JOURNAL))

base = score_de(X)
journal("un trou aléatoire", "gardé tel quel (référence)",
        "le taux d’absence ne dépend pas du fait que la valeur manque — trou non informatif",
        score_de(X) - base)
journal("un trou structurel", "fuite — écartée",
        "le trou suit la réponse : la garder serait prédire avec la réponse",
        score_de(X_garde) - base)
journal("des colonnes inutiles", "écartées",
        "constante, bruit pur, ou redondante avec une autre colonne",
        score_de(X.drop(columns=inutiles)) - base)
journal("un code non documenté", "gardé, en le sachant inexpliqué",
        "aucun dictionnaire ne dit ce que valent 7 et 9 ; le retirer serait décider sans savoir",
        sans_code - base)
journal("une valeur impossible", "lignes retirées",
        "un âge de −1 n’est pas un âge : la valeur est fausse, pas manquante",
        sans_age - base)
journal("des lignes dupliquées", "vérifié, rien à faire",
        "aucune ligne dupliquée dans cette table (mesuré)",
        sans_double - base)


            décision  ... > double de la barre ?
0  un trou aléatoire  ...                    non

[1 rows x 5 columns]
             décision  ... > double de la barre ?
0   un trou aléatoire  ...                    non
1  un trou structurel  ...                    oui

[2 rows x 5 columns]
                décision  ... > double de la barre ?
0      un trou aléatoire  ...                    non
1     un trou structurel  ...                    oui
2  des colonnes inutiles  ...                    non

[3 rows x 5 columns]
                décision  ... > double de la barre ?
0      un trou aléatoire  ...                    non
1     un trou structurel  ...                    oui
2  des colonnes inutiles  ...                    non
3  un code non documenté  ...                    non

[4 rows x 5 columns]
                décision  ... > double de la barre ?
0      un trou aléatoire  ...                    non
1     un trou structurel  ...                    oui
2  des colonnes inutiles  ..

> **Question →** `Devoir_maison_2.ipynb`, **question 8 (carnet J3) (2/2)** — le journal de nettoyage, tel que vous le rendez.


In [24]:
# le pipeline : de la table brute à la prédiction, en un seul endroit
# Même contrat que `predict` : une table brute entre, un 0/1 par ligne sort.
def nettoyer(brut):
    """Les décisions apprises sur la table d’apprentissage, appliquées à une table brute :
       colonnes inutiles écartées, colonnes trop fines écartées, et les catégories de
       l’apprentissage — jamais recalculées sur la table qu’on reçoit."""
    d = matrice(brut)[0]
    d = d.drop(columns=[c for c in inutiles if c in d.columns])
    d = d[[c for c in X.columns if c in d.columns]]
    for c in CAT:
        if c not in d.columns:
            continue
        d[c] = d[c].astype("category").cat.set_categories(X[c].cat.categories)
    return d

def pipeline(brut):
    """Reçoit une table brute et rend un tableau de 0/1 — la chaîne complète."""
    return modele_final.predict(nettoyer(brut)).astype(int)

X_propre = nettoyer(train)
modele_final = HistGradientBoostingClassifier(random_state=0).fit(X_propre, y)
print("prédictions sur 500 lignes brutes :", pipeline(test.head(500)).shape)
print(f"la table préparée : {X_propre.shape[1]} colonnes (les {len(inutiles)} inutiles sont parties)")

soumission = pd.DataFrame({ID: test[ID], CIBLE: pipeline(test)})
soumission.to_csv("submission.csv", index=False)
print("le fichier écrit : submission.csv — le nom que la plateforme attend \n")
print("dimensions :", soumission.shape, "(lignes, colonnes)")


prédictions sur 500 lignes brutes : (500,)
la table préparée : 21 colonnes (les 5 inutiles sont parties)
le fichier écrit : submission.csv — le nom que la plateforme attend 

dimensions : (63721, 2) (lignes, colonnes)


> **Question →** `Devoir_maison_2.ipynb`, **question 9 (carnet J3)** — le pipeline, et sa solidité.


### Et une chose qu’on ne traite pas encore : le temps

Nos deux tableaux ne sont pas tirés au hasard dans une population. Le fichier est un
**historique** : le test vient après l’entraînement dans le temps. Personne ne nous l’a dit. Il
faut le voir dans les dates, et le mesurer. C’est ce que fait la cellule qui suit — y compris là, y compris là
où le découpage n’est pas parfait.

Pourquoi ça compte : un découpage **au hasard** sur un historique laisse le futur entrer dans
l’apprentissage. Le score monte, et il ne voudra rien dire. On y revient à la séance 5 —
« validation croisée et validation temporelle » : c’est là qu’on apprendra à découper sans se
mentir.

Pour plus tard : comment découper sans se mentir.
Le découpage temporel s’apprend, il ne se devine pas.


In [25]:
# la preuve, sans modèle : où s'arrête l'historique, où commence le test ?
historique = pd.to_datetime(train["date_rdv"])
a_venir = pd.to_datetime(test["date_rdv"])

print("la période de l'historique \n")
print("  train : du", historique.min().date(), "au", historique.max().date())
print("  test  : du", a_venir.min().date(), "au", a_venir.max().date())

# le découpage est chronologique — mais jusqu'où, exactement ?
borne = historique.max()
dedans = a_venir <= borne
apres = a_venir[~dedans]
print(f"\n→ {100 * (~dedans).mean():.2f} % des lignes de test viennent après l'historique :",
      f"du {apres.min().date()} au {apres.max().date()}")
print(f"  {int(dedans.sum())} ligne(s) de test portent une date antérieure à la borne.")
print()
print("Ces quelques lignes ne sont pas un détail à balayer : un découpage chronologique se vérifie,")
print("il ne se suppose pas. Elles ne changent rien à la leçon — un découpage au hasard, lui,")
print("mélangerait les deux périodes entièrement, et le score ne voudrait plus rien dire — mais")
print("elles disent exactement où le découpage est imparfait, et c'est ce qu'on attend d'un contrôle.")


la période de l'historique 

  train : du 2023-01-02 au 2025-11-18
  test  : du 2023-01-04 au 2026-12-30

→ 99.50 % des lignes de test viennent après l'historique : du 2025-12-31 au 2026-12-30
  317 ligne(s) de test portent une date antérieure à la borne.

Ces quelques lignes ne sont pas un détail à balayer : un découpage chronologique se vérifie,
il ne se suppose pas. Elles ne changent rien à la leçon — un découpage au hasard, lui,
mélangerait les deux périodes entièrement, et le score ne voudrait plus rien dire — mais
elles disent exactement où le découpage est imparfait, et c'est ce qu'on attend d'un contrôle.


### Ce qu’on retient

- **Un score se lit contre un concurrent nommé** : la réponse constante d’abord, la règle simple
  en place ensuite. Un écart se juge contre la **barre de bruit**, pas à l’œil.
- **L’affichage est un petit échantillon** : sur 1 000 lignes, l’erreur-type d’une exactitude vaut
  plus d’un point. Le classement qu’il donne ne se retrouve pas sur des lignes que personne n’a
  regardées.
- **Le tableau à quatre colonnes** (modèle · validation · affiché · écart) est ce qui rend l’écart
  lisible. Une colonne vide dit qu’un modèle n’a pas été envoyé : ce n’est pas un oubli.
- **Un plafond existe**, et il n’est pas dans le modèle : trois familles très différentes donnent
  ici presque le même chiffre. On le mesure avant de chercher à l’atteindre.
- Un trou n’est pas toujours un trou. Aléatoire, il ne vaut rien. Structuré, il porte de
  l’information — mais s’il **suit la réponse**, c’est une fuite : on l’écarte, et le gain
  qu’il affiche n’est pas un gain.
- Le nettoyage change rarement le score. Il change ce qu’on peut défendre, et ce qui survit quand
  les données changent. **Le journal de nettoyage** est ce qu’on rend.
- **Les deux tableaux ne sont pas tirés au hasard dans une population** : le fichier est un
  historique, et le test vient après. Un découpage au hasard y laisserait entrer le futur.

**Ce qu’on laisse ouvert :** comparer deux modèles jugés sur les mêmes lignes. La comparaison se fait ligne par ligne — le test a un nom (McNemar).
La **calibration** des probabilités annoncées. Et le **découpage temporel**, qu’on ne sait pas
encore écrire sans se mentir.
